<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/twelvelabs-io/twelvelabs-developer-experience/blob/main/quickstarts/TwelveLabs_Quickstart_Segment.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in  Colab</a>
  </td>
</table>

# Segment videos

This guide shows how to use the TwelveLabs Python SDK to segment videos into structured, timestamped data.

## Key concepts

- **Asset**: Your uploaded content
- **Analysis task**: An asynchronous operation for processing your video. Contains a status and the results when complete.
- **Segment definition**: A description of a type of segment you want to extract. Each definition includes a unique identifier, a natural language description, and optional custom fields.
- **Segment field**: A custom metadata field to extract for each segment. Each field has a name, a type, and a description.

## How it works

To segment a video, upload it as an asset, then create an asynchronous analysis task with Pegasus 1.5. Set the analysis mode to time-based metadata and provide one or more segment definitions. The platform identifies the segment boundaries and returns custom metadata for each segment as a JSON-encoded string.

The upload method in this guide supports public video URLs up to 2 GB and local files up to 200 MB. For details about the available upload methods and the corresponding limits, see the [Upload methods](https://docs.twelvelabs.io/docs/concepts/upload-methods) page.

**Customize segmentation**

You can customize segmentation in the following ways:
- Submit up to 10 segment definitions in a single request to extract different types of segments
- Define up to 20 custom fields per segment definition, each typed as string, boolean, number, integer, or array
- List the allowed values for a field with `enum` to constrain the output
- Set minimum and maximum segment durations to control the segment boundaries
- Add up to 4 reference images to a segment definition to provide visual context for detection

# Prerequisites

- To use the platform, you need an API key:
  1. If you don't have an account, [sign up](https://playground.twelvelabs.io/) for a free account. No credit card is required to use the Free plan. This plan allows you to index up to 600 minutes of videos, which is sufficient for a small project.
  2. Go to the [API Keys](https://playground.twelvelabs.io/dashboard/api-keys) page.
  3. If you need to create a new key, select the **Create API Key** button. Enter a name and set the expiration period. The default is 12 months.
  4. Select the **Copy** icon next to your key to copy it to your clipboard.
- Your video files must meet the [format requirements](https://docs.twelvelabs.io/docs/concepts/models/pegasus#input-requirements).

# Procedure

## Install the TwelveLabs Python SDK

In [ ]:
%pip install twelvelabs

## Import the required packages

In [ ]:
import json
import time
from twelvelabs import TwelveLabs
from twelvelabs.types import AsyncResponseFormat, VideoContext_AssetId

## Configure your API key


In [ ]:
# For Google Colab, store your API key as a Secret named `TL_API_KEY`. If you don't know how to create a Colab Secret, see https://medium.com/@parthdasawant/how-to-use-secrets-in-google-colab-450c38e3ec75.

from google.colab import userdata
TL_API_KEY = userdata.get("TL_API_KEY")

# For other Python environments, you can use environment variables
# TL_API_KEY = os.environ.get('TL_API_KEY')

## Upload a video

In [ ]:
client = TwelveLabs(api_key=TL_API_KEY)

asset = client.assets.create(
    method="url",
    url="<YOUR_VIDEO_URL>" # Example: https://github.com/twelvelabs-io/twelvelabs-developer-experience/raw/refs/heads/main/quickstarts/steve_jobs_introduces_iphone_in_2007.mp4
    # Or use method="direct" and file=open("<PATH_TO_VIDEO_FILE>", "rb") to upload a file from the local file system
)
print(f"Created asset: id={asset.id}")

## Check the status of the asset

In [ ]:
print("Waiting for asset to be ready...")
while True:
    asset = client.assets.retrieve(asset.id)
    if asset.status == "ready":
        print("Asset is ready")
        break
    if asset.status == "failed":
        raise RuntimeError(f"Asset processing failed: id={asset.id}")
    time.sleep(5)

## Create a video segmentation task

Define the segments and fields you want to extract, then create an asynchronous task. This example segments the video into scenes and extracts three fields for each scene. This operation is asynchronous.

In [ ]:
video = VideoContext_AssetId(asset_id=asset.id)
task = client.analyze_async.tasks.create(
    video=video,
    model_name="pegasus1.5",
    analysis_mode="time_based_metadata",
    response_format=AsyncResponseFormat(
        type="segment_definitions",
        segment_definitions=[
            {
                "id": "scenes",
                "description": "Segment the video into distinct scenes based on changes in setting, topic, or visual composition",
                "fields": [
                    {
                        "name": "sentiment",
                        "type": "string",
                        "description": "The overall sentiment of this scene",
                        "enum": ["positive", "negative", "neutral"]
                    },
                    {
                        "name": "key_objects",
                        "type": "array",
                        "description": "Notable objects visible in the scene",
                        "items": {"type": "string"}
                    },
                    {
                        "name": "contains_speech",
                        "type": "boolean",
                        "description": "Whether the scene contains speech or dialogue"
                    }
                ]
            }
        ]
    )
)
print(f"Task ID: {task.task_id}")

## Monitor the status

In [ ]:
while True:
    task = client.analyze_async.tasks.retrieve(task.task_id)
    if task.status == "ready":
        print("Task completed")
        break
    elif task.status == "failed":
        print("Task failed")
        break
    else:
        print("Task still processing...")
        time.sleep(5)

## Process the results

In [ ]:
data = json.loads(task.result.data)
for segment in data["scenes"]:
    print(f"\n[{segment['start_time']:.1f}s - {segment['end_time']:.1f}s]")
    meta = segment["metadata"]
    print(f"  Sentiment: {meta['sentiment']}")
    print(f"  Key objects: {', '.join(meta['key_objects'])}")
    print(f"  Contains speech: {meta['contains_speech']}")

# Next steps

For a comprehensive guide, see the [Segment videos](https://docs.twelvelabs.io/docs/guides/segment-videos) page.